# Model 2: Fine-tuned DistilBERT (End-to-End)

**Architecture:** DistilBERT (66M params, fine-tuned) → [CLS] embedding (768-dim) → Regression head (LayerNorm → Linear(256) → GELU → Linear(1))

**Target:** Beat both BoW DNN ($46.49) and SentenceTransformer DNN (Model 1)

**Key techniques:**
- Discriminative LR: encoder 2e-5, head 1e-4
- Mixed precision (fp16)
- Linear warmup 1000 steps
- Early stopping patience=2
- max_length=128 (covers 99.8% of summaries)

## vast.ai Setup (chỉ chạy lần đầu khi thuê máy)

Sau khi `git clone` repo và `cd` vào đúng thư mục, mở terminal trên vast.ai và chạy:

```bash
pip install uv
uv sync
```

Sau đó khởi động lại Jupyter kernel rồi chạy các cell bên dưới.

In [1]:
from pricer.items import Item
from pricer.distilbert_model import DistilBERTRunner
from pricer.evaluator import evaluate, plot_training_history

## 1. Load Data

In [2]:
train, val, test = Item.from_hub("SeanSunny/items_full")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

Train: 800,000 | Val: 10,000 | Test: 10,000


## 2. Setup Model

Loads DistilBERT pretrained weights, creates tokenized DataLoaders, configures discriminative LR optimizer.

In [3]:
runner = DistilBERTRunner(train, val[:1000])
runner.setup(batch_size=32)

Loading DistilBERT tokenizer...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Loading DistilBERT model...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

DistilBERT Regressor: 66,561,537 params (encoder: 66,362,880, head: 198,657)
Using cuda


## 3. Train

Max 5 epochs with early stopping (patience=2). Mixed precision (fp16) on CUDA. Linear warmup 1000 steps.

In [4]:
history = runner.train(epochs=5, patience=2, warmup_steps=1000)

Epoch 1/5:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch [1/5]
  Train Loss: 0.4496, Val Loss: 0.3958
  Val MAE: $54.60, LR: 0.00001613
  ** New best Val MAE: $54.60


Epoch 2/5:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch [2/5]
  Train Loss: 0.3707, Val Loss: 0.3741
  Val MAE: $51.12, LR: 0.00001210
  ** New best Val MAE: $51.12


Epoch 3/5:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch [3/5]
  Train Loss: 0.3321, Val Loss: 0.3545
  Val MAE: $49.22, LR: 0.00000806
  ** New best Val MAE: $49.22


Epoch 4/5:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch [4/5]
  Train Loss: 0.3031, Val Loss: 0.3514
  Val MAE: $48.38, LR: 0.00000403
  ** New best Val MAE: $48.38


Epoch 5/5:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch [5/5]
  Train Loss: 0.2821, Val Loss: 0.3489
  Val MAE: $47.42, LR: 0.00000000
  ** New best Val MAE: $47.42


## 4. Training History

In [5]:
plot_training_history(history, title="Fine-tuned DistilBERT")

## 5. Evaluate on 200 Test Samples

Using `evaluate()` from `pricer/evaluator.py` — same evaluation framework as all other models.

In [6]:
evaluate(runner.inference, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$87 $91 $15 $25 $90 $74 $42 $7 $13 $5 $12 $144 $5 $4 $3 $1 $46 $15 $3 $49 $42 $18 $13 $148 $61 $212 $205 $4 $55 $58 $21 $6 $108 $15 $26 $317 $84 $27 $50 $5 $1 $45 $1 $169 $109 $4 $2 $3 $92 $0 $13 $38 $170 $36 $18 $21 $15 $118 $12 $7 $119 $46 $34 $42 $364 $7 $26 $298 $5 $32 $16 $1 $11 $7 $21 $2 $90 $2 $3 $1 $22 $13 $7 $58 $16 $151 $139 $173 $13 $13 $21 $1 $1 $2 $1 $74 $6 $3 $50 $197 $10 $6 $5 $48 $21 $77 $0 $291 $14 $46 $27 $8 $13 $40 $20 $51 $22 $3 $66 $98 $8 $70 $16 $17 $32 $19 $3 $31 $42 $67 $66 $21 $17 $2 $73 $0 $93 $35 $32 $10 $4 $19 $11 $10 $79 $10 $18 $290 $52 $6 $1 $22 $7 $24 $16 $68 $34 $5 $75 $6 $86 $15 $8 $2 $140 $3 $119 $26 $2 $0 $38 $18 $227 $16 $16 $7 $3 $19 $60 $14 $197 $7 $56 $30 $6 $17 $74 $13 $16 $11 $8 $12 $2 $75 $1 $11 $9 $6 $15 $11 

## 6. Save Model Weights

In [7]:
runner.save("distilbert_model.pth")
print("Saved to distilbert_model.pth")

Saved to distilbert_model.pth


# 1. Sanity check — inference trên trained runner
sample = test[0]
pred_original = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred_original:.2f}")
print(f"Error:   ${abs(pred_original - sample.price):.2f}")
print()

# 2. Load roundtrip test — load lại từ .pth và so sánh kết quả
runner.load("distilbert_model.pth")
pred_loaded = runner.inference(sample)
diff = abs(pred_original - pred_loaded)
assert diff < 0.01, f"Load mismatch! Before=${pred_original:.2f} After=${pred_loaded:.2f}"
print(f"Load roundtrip test PASSED. Diff: ${diff:.4f}")

In [8]:
# Quick sanity check
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred:.2f}")
print(f"Error:   ${abs(pred - sample.price):.2f}")

Product: Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal
Actual:  $219.00
Predict: $306.20
Error:   $87.20
